# 03 – SARIMA Model

**PM2.5 Time Series: Factor Analysis & SARIMA**  
Notebook này triển khai **Step 3 – SARIMA** dựa trên logic trong `src/sarima_model.py`.

## Mục tiêu

- Đọc `data/processed/fa_data.csv` (đã có PM2.5 và Factor1, Factor2, Factor3).
- Gộp dữ liệu theo **ngày** (trung bình theo ngày) để dùng SARIMA theo ngày, mùa 7 ngày.
- **Phân rã chuỗi** (Trend, Seasonal, Residual) và kiểm định **stationarity** (ADF).
- Vẽ **ACF/PACF** để hỗ trợ chọn tham số.
- Dùng **auto_arima** (pmdarima) với biến ngoại sinh (Factor1–3), chia train/test, fit và lưu mô hình.

## Các lệnh / bước chính

| Bước | Lệnh / Hàm | Mô tả ngắn |
|------|------------|------------|
| 1 | `pd.read_csv(..., parse_dates=True)` | Đọc fa_data.csv, index là datetime |
| 2 | `df.resample("D").mean()` | Gộp theo ngày (daily baseline) |
| 3 | `seasonal_decompose(..., period=7)` | Phân rã: Observed = Trend + Seasonal + Residual |
| 4 | `adfuller(series)` | Kiểm định ADF (chuỗi dừng hay không) |
| 5 | `plot_acf`, `plot_pacf` | ACF/PACF để gợi ý p, q |
| 6 | Train/test split | 80% train, 20% test (theo thời gian) |
| 7 | `auto_arima(..., X=exog, seasonal=True, m=7)` | Tìm (p,d,q)(P,D,Q)m và fit |
| 8 | `joblib.dump(...)` | Lưu mô hình + exog_cols vào data/processed |

## Điều kiện trước khi chạy

- Đã có `data/processed/fa_data.csv` (tạo từ Step 2 / `02_factor_analysis.ipynb` hoặc `python src/factor_analysis.py`).

## Thiết lập môi trường

- Thêm project root vào `sys.path` để có thể `import` từ `src`.
- Định nghĩa thư mục: `PROCESSED_DIR` (fa_data), `ARIMA_FIG` (figures), `TABLES_DIR` (bảng ADF).
- Các thư viện: `pandas`, `matplotlib`, `statsmodels` (decompose, adfuller, acf/pacf), `pmdarima` (auto_arima), `joblib` (lưu mô hình).

In [ ]:
# Thiết lập môi trường
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from pmdarima import auto_arima
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

# Project root (chạy từ notebooks/ hoặc từ root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROCESSED_DIR = ROOT / "data" / "processed"
ARIMA_FIG = ROOT / "reports" / "figures" / "03_sarima"
TABLES_DIR = ROOT / "reports" / "tables"

for p in [PROCESSED_DIR, ARIMA_FIG, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
pd.set_option("display.max_columns", 50)

# Tên cột biến ngoại sinh (đúng với fa_data)
FACTOR_COLS = ["Factor1", "Factor2", "Factor3"]

print("ROOT:", ROOT)
print("fa_data:", PROCESSED_DIR / "fa_data.csv")
print("Figures:", ARIMA_FIG)

## 1. Đọc dữ liệu fa_data

**Lệnh:** `pd.read_csv(path, index_col=0, parse_dates=True)`  
- `index_col=0`: cột đầu tiên là index (datetime).  
- `parse_dates=True`: chuyển index thành kiểu datetime.  

File `fa_data.csv` chứa dữ liệu theo **giờ**, gồm PM2.5 và các nhân tố Factor1, Factor2, Factor3 (từ Factor Analysis).

In [ ]:
fa_path = PROCESSED_DIR / "fa_data.csv"
if not fa_path.exists():
    raise FileNotFoundError(f"Thiếu {fa_path}. Hãy chạy Step 2 (Factor Analysis) trước.")

df = pd.read_csv(fa_path, index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Index:", df.index.min(), "->", df.index.max())
print("Cột:", list(df.columns))
df.head(10)

## 2. Gộp theo ngày (aggregate to daily)

**Lệnh:** `df[numeric_cols].resample("D").mean()`  
- Chỉ lấy cột số, gộp theo **ngày** ("D"), lấy **trung bình** mỗi ngày.  
- Mục đích: SARIMA baseline dùng **daily** data, chu kỳ mùa **m = 7** (tuần).  
- Sau bước này mỗi dòng là một ngày.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
daily = df[numeric_cols].resample("D").mean()
daily = daily.dropna(how="all")

print("Shape daily:", daily.shape)
print("Số ngày:", len(daily))
if "PM2.5" not in daily.columns:
    raise ValueError("Cột PM2.5 không có trong fa_data")

pm25 = daily["PM2.5"]
daily.head(10)

## 3. Phân rã chuỗi thời gian (Decomposition)

**Lệnh:** `seasonal_decompose(series, model="additive", period=7, extrapolate_trend="freq")`  
- **model="additive"**: Observed = Trend + Seasonal + Residual.  
- **period=7**: mùa 7 ngày (tuần).  
- **extrapolate_trend="freq"**: ngoại suy trend ở hai đầu để tránh NaN.  

Kết quả: 4 chuỗi — Observed, Trend, Seasonal, Residual. Lưu figure vào `reports/figures/03_arima/decomposition.png`.

In [ ]:
PERIOD = 7  # chu kỳ mùa (ngày)
decomp = seasonal_decompose(pm25, model="additive", period=PERIOD, extrapolate_trend="freq")

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomp.observed.plot(ax=axes[0], color="#2E86AB")
axes[0].set_ylabel("Observed")
axes[0].set_title("PM2.5 Time Series Decomposition (Daily)")
decomp.trend.plot(ax=axes[1], color="#E94F37")
axes[1].set_ylabel("Trend")
decomp.seasonal.plot(ax=axes[2], color="#44AF69")
axes[2].set_ylabel("Seasonal")
decomp.resid.plot(ax=axes[3], color="#8B8B8B")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()

decomp_path = ARIMA_FIG / "decomposition.png"
plt.savefig(decomp_path, dpi=300, bbox_inches="tight")
plt.show()
print("Đã lưu:", decomp_path)

## 4. Kiểm định ADF (Augmented Dickey–Fuller)

**Lệnh:** `adfuller(series.dropna(), autolag="AIC")`  
- Kiểm định **stationarity**: H0 = chuỗi không dừng.  
- **p-value < 0.05** → bác bỏ H0 → chuỗi **dừng** (stationary).  
- **autolag="AIC"**: tự chọn số lag tối ưu theo AIC.  

Kết quả lưu vào `reports/tables/adf_results.txt` và `adf_results.csv`.

In [ ]:
adf_result = adfuller(pm25.dropna(), autolag="AIC")

adf_stats = {
    "adf_statistic": adf_result[0],
    "p_value": adf_result[1],
    "usedlag": adf_result[2],
    "nobs": adf_result[3],
    "critical_values": adf_result[4],
    "icbest": adf_result[5],
}

print("ADF Statistic:", f"{adf_stats['adf_statistic']:.6f}")
print("p-value:", f"{adf_stats['p_value']:.6f}")
print("Used lag:", adf_stats["usedlag"])
print("Critical values:", adf_stats["critical_values"])
conclusion = "Chuỗi DỪNG (stationary)" if adf_stats["p_value"] < 0.05 else "Chuỗi KHÔNG dừng (non-stationary)"
print("Kết luận:", conclusion)

# Lưu báo cáo text
lines = [
    "Augmented Dickey-Fuller Test Results",
    "=" * 40,
    f"ADF Statistic: {adf_stats['adf_statistic']:.6f}",
    f"p-value:       {adf_stats['p_value']:.6f}",
    f"Used lag:      {adf_stats['usedlag']}",
    f"N observations: {adf_stats['nobs']}",
    "Critical values:",
]
for k, v in adf_stats["critical_values"].items():
    lines.append(f"  {k}: {v:.3f}")
lines.append("")
lines.append("Conclusion: Series is STATIONARY" if adf_stats["p_value"] < 0.05 else "Conclusion: Series is NON-STATIONARY")

txt_path = TABLES_DIR / "adf_results.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
csv_path = TABLES_DIR / "adf_results.csv"
pd.DataFrame([{"adf_statistic": adf_stats["adf_statistic"], "p_value": adf_stats["p_value"]}]).to_csv(csv_path, index=False)
print("Đã lưu:", txt_path, csv_path)

## 5. ACF và PACF

**Lệnh:** `plot_acf(series, lags=...)`, `plot_pacf(series, lags=..., method="ywm")`  
- **ACF** (Autocorrelation): tương quan của chuỗi với chính nó tại các độ trễ → gợi ý bậc **q** (MA).  
- **PACF** (Partial ACF): tương quan riêng phần → gợi ý bậc **p** (AR).  
- **method="ywm"**: ước lượng PACF (Yule–Walker modified).  

Lưu figure vào `reports/figures/03_arima/acf_pacf.png`.

In [ ]:
LAGS = 40
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(pm25.dropna(), lags=LAGS, ax=axes[0])
axes[0].set_title("Autocorrelation (ACF)")
plot_pacf(pm25.dropna(), lags=LAGS, ax=axes[1], method="ywm")
axes[1].set_title("Partial Autocorrelation (PACF)")
plt.suptitle("ACF & PACF for PM2.5 (Daily)", fontsize=14, y=1.02)
plt.tight_layout()

acf_path = ARIMA_FIG / "acf_pacf.png"
plt.savefig(acf_path, dpi=300, bbox_inches="tight")
plt.show()
print("Đã lưu:", acf_path)

## 6. Chia train / test

**Lệnh:** Chia theo **thời gian** (không xáo trộn): 80% đầu làm train, 20% cuối làm test.  
- `y_train`, `y_test`: chuỗi PM2.5.  
- `exog_train`, `exog_test`: DataFrame các cột Factor1, Factor2, Factor3 (nếu có) dùng làm biến ngoại sinh cho SARIMAX.

In [ ]:
TEST_RATIO = 0.2
n = len(daily)
split_idx = int(n * (1 - TEST_RATIO))
train_df = daily.iloc[:split_idx]
test_df = daily.iloc[split_idx:]

y_train = train_df["PM2.5"]
y_test = test_df["PM2.5"]

exog_cols = [c for c in FACTOR_COLS if c in daily.columns]
exog_train = train_df[exog_cols] if exog_cols else None
exog_test = test_df[exog_cols] if exog_cols else None

print("Train: ", len(y_train), " ngày")
print("Test:  ", len(y_test), " ngày")
print("Biến ngoại sinh (exog):", exog_cols if exog_cols else "Không dùng")

## 7. Auto ARIMA với biến ngoại sinh

**Lệnh:** `auto_arima(y_train, X=exog_train.values, seasonal=True, m=7, stepwise=True, ...)`  
- **X**: ma trận biến ngoại sinh (Factor1, Factor2, Factor3) — phải cùng số hàng với `y_train`.  
- **seasonal=True, m=7**: SARIMA với mùa 7 ngày.  
- **stepwise=True**: tìm (p,d,q)(P,D,Q) nhanh hơn.  
- **suppress_warnings=True, error_action="ignore"**: giảm log khi thử nhiều mô hình.  

Kết quả: mô hình ARIMA đã fit (order + seasonal_order). Có thể mất vài phút.

In [ ]:
exog_arr = exog_train.values if exog_train is not None else None

model = auto_arima(
    y_train,
    X=exog_arr,
    seasonal=True,
    m=PERIOD,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    trace=False,
)

print("Order (p,d,q):", model.order)
print("Seasonal order (P,D,Q,m):", model.seasonal_order)
print("Tóm tắt mô hình:")
model.summary()

## 8. Lưu mô hình

**Lệnh:** `joblib.dump({...}, path)`  
- Lưu dict gồm: **model** (đối tượng ARIMA đã fit), **exog_cols** (tên cột ngoại sinh), **seasonal_period** (m=7).  
- Khi dự báo sau này cần dùng cùng `exog` (Factor1–3) cho từng bước.  

File lưu: `data/processed/sarima_model.joblib`.

In [ ]:
model_path = PROCESSED_DIR / "sarima_model.joblib"
joblib.dump(
    {
        "model": model,
        "exog_cols": FACTOR_COLS,
        "seasonal_period": PERIOD,
    },
    model_path,
)
print("Đã lưu mô hình:", model_path)

# Kiểm tra load lại
loaded = joblib.load(model_path)
print("exog_cols trong file:", loaded["exog_cols"])
print("seasonal_period:", loaded["seasonal_period"])